# Loading Data by Activity Type
Access pattern: `participants[participant_id][activity_type][sensor_type][sensor_location]`
- Example: `participants["2995"]["Perturb"]["Trip"]["ECG"]`
- Example: `participants["2995"]["Perturb"]["Trip"]["IMU"]["Left_Thigh"]`

In [1]:
import os
import pandas as pd
import glob
from collections import defaultdict
import re

# ASSIGN COLUMN NAMES
ECG_COLS   = ["time_s", "ecg_V"]
GSS_COLS   = ["time_s", "gss_V"]
IMU_COLS   = ["time_s", "acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
LABEL_COLS = ["event", "time_s", "other"]

from tkinter import Tk, filedialog
import os
import glob

# Hide tkinter window
Tk().withdraw()

# Choose folder containing CSV files
DATA_DIR = filedialog.askdirectory(title="Select folder with CSV files")

if not DATA_DIR:
    raise RuntimeError("No folder selected!")

print("Loading data from:", DATA_DIR)

# Get all CSV files in the selected folder
all_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))

print(f"Found {len(all_files)} CSV files")

# Categorize activities into ADL (Activities of Daily Living) and Perturb (Perturbation)
# ADL: Activities of Daily Living - normal daily activities
ADL_ACTIVITIES = {"Walk", "Stand", "Sit", "Lie", "Stairs", "Pick", "Jog", "JJ"}
# Perturb: Perturbations to induce fall or near fall
PERTURB_ACTIVITIES = {"Slip", "Trip", "Miss", "Hit", "Coll", "LOS", "ITD", "ITR"}

def make_activity_dict():
    return {
        "IMU": {"Back": None, "Left_Thigh": None, "Right_Thigh": None},
        "ECG": None,
        "GSS": None,
        "labels": None
    }

def make_category_dict():
    return {
        "ADL": defaultdict(make_activity_dict),
        "Perturb": defaultdict(make_activity_dict)
    }

# Structure: participants[participant_id]["ADL" or "Perturb"][activity_type][sensor_type][sensor_location]
participants = defaultdict(make_category_dict)

# Track trial identifiers for each participant/activity to keep the latest one
trial_tracker = defaultdict(lambda: defaultdict(list))  # {pid: {activity: [(trial_id, sensor, file)]}}
label_tracker = defaultdict(lambda: defaultdict(list))  # {pid: {activity: [(suffix, file)]}}

SENSOR_SUFFIXES = ["Left_Thigh", "Right_Thigh", "Back", "ECG", "GSS"]

def categorize_activity(activity):
    """Determine if activity is ADL or Perturb"""
    if activity in ADL_ACTIVITIES:
        return "ADL"
    elif activity in PERTURB_ACTIVITIES:
        return "Perturb"
    else:
        # Default to ADL for unknown activities
        return "ADL"

def parse_data_filename(base):
    """Parse data files like: data_P2995_T06_Trip_Left_Thigh.csv or data_P3033_T01A_Trip_Back.csv"""
    if not (base.startswith("data_P") and base.endswith(".csv")):
        return None
    stem = base[:-4]
    parts = stem.split("_")
    if len(parts) < 4:
        return None

    pid_part = parts[1]  # "P2995"
    if not pid_part.startswith("P") or not pid_part[1:].isdigit():
        return None
    pid = pid_part[1:]

    trial_part = parts[2]  # "T06" or "T01A" or "T01B"
    # Use regex to extract trial number and optional suffix
    match = re.match(r'T(\d+)([A-Z]?)', trial_part)
    if not match:
        return None
    trial_num = int(match.group(1))
    trial_suffix = match.group(2) if match.group(2) else ''
    trial_id = (trial_num, trial_suffix)  # e.g., (1, 'A') or (1, 'B') or (6, '')
    
    # Extract activity and sensor from remainder
    remainder = "_".join(parts[3:])
    sensor = None
    activity = None
    for sfx in SENSOR_SUFFIXES:
        tag = "_" + sfx
        if remainder.endswith(tag):
            sensor = sfx
            activity = remainder[: -len(tag)]
            break

    if sensor is None or not activity:
        return None
    return pid, trial_id, activity, sensor

def parse_label_filename(base):
    """Parse label files like: labels_P2995_Trip.csv or labels_P5690_Coll_A.csv"""
    if not (base.startswith("labels_P") and base.endswith(".csv")):
        return None
    stem = base[:-4]
    rest = stem[len("labels_P"):]  # "2995_Trip" or "5690_Coll_A"
    if "_" not in rest:
        return None
    pid_str, activity_part = rest.split("_", 1)
    if not pid_str.isdigit() or not activity_part:
        return None
    
    # Check if activity has A/B suffix (e.g., "Coll_A" or "Coll_B")
    suffix = ''
    activity = activity_part
    if activity_part.endswith('_A'):
        activity = activity_part[:-2]
        suffix = 'A'
    elif activity_part.endswith('_B'):
        activity = activity_part[:-2]
        suffix = 'B'
    
    return pid_str, activity, suffix

# First pass: collect all files and track trial identifiers
file_info = []

for f in all_files:
    base = os.path.basename(f)
    
    if os.path.getsize(f) == 0:
        continue
    
    # Label files
    lab = parse_label_filename(base)
    if lab is not None:
        pid, activity, suffix = lab
        label_tracker[pid][activity].append((suffix, f))
        continue
    
    # Data files
    parsed = parse_data_filename(base)
    if parsed is not None:
        pid, trial_id, activity, sensor = parsed
        trial_tracker[pid][activity].append((trial_id, sensor, f))
        file_info.append((pid, trial_id, activity, sensor, f))

# Second pass: load data, keeping only the highest trial identifier for each activity
for pid in trial_tracker:
    for activity in trial_tracker[pid]:
        # Find the maximum trial identifier (highest number, then highest suffix)
        # trial_id is a tuple: (trial_num, suffix) e.g., (1, 'B') > (1, 'A') > (1, '')
        max_trial_id = max(trial_id for trial_id, _, _ in trial_tracker[pid][activity])
        
        # Determine category (ADL or Perturb)
        category = categorize_activity(activity)
        
        # Load only files from the max trial
        for trial_id, sensor, f in trial_tracker[pid][activity]:
            if trial_id == max_trial_id:
                if sensor in ["Back", "Left_Thigh", "Right_Thigh"]:
                    df = pd.read_csv(f, header=None, names=IMU_COLS)
                    participants[pid][category][activity]["IMU"][sensor] = df
                elif sensor == "ECG":
                    df = pd.read_csv(f, header=None, names=ECG_COLS)
                    participants[pid][category][activity]["ECG"] = df
                elif sensor == "GSS":
                    df = pd.read_csv(f, header=None, names=GSS_COLS)
                    participants[pid][category][activity]["GSS"] = df

# Third pass: load label files (keep only the last version - B over A over no suffix)
for pid in label_tracker:
    for activity in label_tracker[pid]:
        # Sort by suffix: '' < 'A' < 'B', take the last one
        label_tracker[pid][activity].sort(key=lambda x: x[0])
        max_suffix, label_file = label_tracker[pid][activity][-1]
        
        # Determine category (ADL or Perturb)
        category = categorize_activity(activity)
        
        # Only load if the participant/activity already exists (has sensor data)
        if pid in participants and activity in participants[pid][category]:
            df = pd.read_csv(label_file, header=None, names=LABEL_COLS)
            participants[pid][category][activity]["labels"] = df

print(f"✅ Loaded data for {len(participants)} participants")
print(f"\nExample participants: {list(participants.keys())[:5]}")

# Count activities by category
adl_count = sum(len(participants[pid]["ADL"]) for pid in participants)
perturb_count = sum(len(participants[pid]["Perturb"]) for pid in participants)
print(f"\nTotal ADL activities: {adl_count}")
print(f"Total Perturb activities: {perturb_count}")

if '2995' in participants:
    print(f"\nParticipant 2995:")
    print(f"  ADL activities: {list(participants['2995']['ADL'].keys())}")
    print(f"  Perturb activities: {list(participants['2995']['Perturb'].keys())}")

Loading data from: /Users/chloechristensen/Library/Mobile Documents/com~apple~CloudDocs/School/Y5/BMEG 400K/Grand_Challenge/data
Found 1281 CSV files
✅ Loaded data for 14 participants

Example participants: ['5923', '4616', '5690', '7202', '7465']

Total ADL activities: 112
Total Perturb activities: 112

Participant 2995:
  ADL activities: ['Walk', 'Sit', 'Pick', 'Stairs', 'Lie', 'Jog', 'Stand', 'JJ']
  Perturb activities: ['Miss', 'ITD', 'Trip', 'Slip', 'LOS', 'Coll', 'ITR', 'Hit']


In [5]:
# Complete overview of all participants with ADL/Perturb breakdown
print("=" * 80)
print("COMPLETE PARTICIPANT OVERVIEW")
print("=" * 80)

for pid in sorted(participants.keys()):
    print(f"\n📊 Participant {pid}:")
    print("-" * 80)
    
    # ADL activities
    if len(participants[pid]["ADL"]) > 0:
        print(f"  ADL Activities ({len(participants[pid]['ADL'])}):")
        for activity in sorted(participants[pid]["ADL"].keys()):
            # Check if all sensors are available
            has_ecg = participants[pid]["ADL"][activity]["ECG"] is not None
            has_gss = participants[pid]["ADL"][activity]["GSS"] is not None
            has_imu = all(participants[pid]["ADL"][activity]["IMU"][s] is not None 
                         for s in ["Back", "Left_Thigh", "Right_Thigh"])
            has_labels = participants[pid]["ADL"][activity]["labels"] is not None
            
            status = "✓" if (has_ecg and has_gss and has_imu) else "⚠"
            label_status = "✓" if has_labels else "✗"
            
            print(f"    {activity:12s} [Sensors: {status}] [Labels: {label_status}]")
    
    # Perturb activities
    if len(participants[pid]["Perturb"]) > 0:
        print(f"  Perturb Activities ({len(participants[pid]['Perturb'])}):")
        for activity in sorted(participants[pid]["Perturb"].keys()):
            # Check if all sensors are available
            has_ecg = participants[pid]["Perturb"][activity]["ECG"] is not None
            has_gss = participants[pid]["Perturb"][activity]["GSS"] is not None
            has_imu = all(participants[pid]["Perturb"][activity]["IMU"][s] is not None 
                         for s in ["Back", "Left_Thigh", "Right_Thigh"])
            has_labels = participants[pid]["Perturb"][activity]["labels"] is not None
            
            status = "✓" if (has_ecg and has_gss and has_imu) else "⚠"
            label_status = "✓" if has_labels else "✗"
            
            print(f"    {activity:12s} [Sensors: {status}] [Labels: {label_status}]")

print("\n" + "=" * 80)
print(f"✅ Total: {len(participants)} participants")
print(f"   - Total ADL activities: {sum(len(participants[pid]['ADL']) for pid in participants)}")
print(f"   - Total Perturb activities: {sum(len(participants[pid]['Perturb']) for pid in participants)}")
print("=" * 80)

COMPLETE PARTICIPANT OVERVIEW

📊 Participant 1960:
--------------------------------------------------------------------------------
  ADL Activities (8):
    JJ           [Sensors: ✓] [Labels: ✗]
    Jog          [Sensors: ✓] [Labels: ✗]
    Lie          [Sensors: ✓] [Labels: ✗]
    Pick         [Sensors: ✓] [Labels: ✗]
    Sit          [Sensors: ✓] [Labels: ✗]
    Stairs       [Sensors: ✓] [Labels: ✗]
    Stand        [Sensors: ✓] [Labels: ✗]
    Walk         [Sensors: ✓] [Labels: ✗]
  Perturb Activities (8):
    Coll         [Sensors: ✓] [Labels: ✓]
    Hit          [Sensors: ✓] [Labels: ✓]
    ITD          [Sensors: ✓] [Labels: ✓]
    ITR          [Sensors: ⚠] [Labels: ✓]
    LOS          [Sensors: ✓] [Labels: ✓]
    Miss         [Sensors: ✓] [Labels: ✓]
    Slip         [Sensors: ✓] [Labels: ✓]
    Trip         [Sensors: ✓] [Labels: ✓]

📊 Participant 2070:
--------------------------------------------------------------------------------
  ADL Activities (8):
    JJ           [Sensor

# Exploring the data

Q3: Plot some examples of a fall, near-fall, and activity of daily living for each of the sensors. Make sure to note which participant and which activities the plots come from

Q4: What sources of error do you expect to encounter for each sensor in measuring falls, near-falls, and activities of daily living? Use plots to demonstrate some of these errors (again, note which participant and which activities the plots come from).

Q6: Given your understanding of the data and looking at the data itself, what information do you think will be most relevant in developing a system that can identify a fall or near-fall? (e.g. what data do you think you will rely on when developing your system and why?) Again, use some plots the justify your answer here and indicate the participant and activity. [15]

In [ ]:
# Go through all participants and check which perturbations have a "4" in the label data
print("=" * 80)
print(f"{'PARTICIPANT':<15} {'ACTIVITY':<15} {'HAS 4?':<10}")
print("=" * 80)

found_count = 0

for pid in sorted(participants.keys()):
    # Check if Perturb category exists and has activities
    if "Perturb" in participants[pid]:
        for activity in sorted(participants[pid]["Perturb"].keys()):
            # Check if labels are loaded
            if participants[pid]["Perturb"][activity]["labels"] is not None:
                labels_df = participants[pid]["Perturb"][activity]["labels"]
                
                # Check for "4" in any column
                has_4 = False
                for col in labels_df.columns:
                    if labels_df[col].astype(str).str.contains('4', na=False).any():
                        has_4 = True
                        break
                
                status = "✅" if has_4 else "❌"
                if has_4:
                    found_count += 1
                    
                print(f"{pid:<15} {activity:<15} {status}")

print("=" * 80)
print(f"Total perturbation trials with '4' in label data: {found_count}")
print("=" * 80)

In [ ]:
# plot the particiapnts with sit against those two ITR and ITD 
# Compare the walk and jog with trip
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from tkinter import Tk, filedialog


# # =========================
# # SAVE LOCATION (your path)
# # =========================
Tk().withdraw()

# Ask user to choose a folder
BASE_DIR = filedialog.askdirectory(title="Select folder to save plots")

if not BASE_DIR:
    raise RuntimeError("No folder selected!")

COMPARE_DIR = os.path.join(BASE_DIR, "Comparisons_3col")
os.makedirs(COMPARE_DIR, exist_ok=True)

print("Saving plots to:", COMPARE_DIR)

# =========================
# HELPERS
# =========================
def safe_name(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^A-Za-z0-9_\-]+", "", s)
    return s

def pid_str(pid) -> str:
    try:
        return f"P{int(pid):03d}"
    except Exception:
        return f"P_{safe_name(pid)}"

def find_activity(pid, activity):
    """Return (kind, data) where kind is 'Perturb' or 'ADL', else (None, None)."""
    if pid not in participants:
        return None, None
    if activity in participants[pid].get("Perturb", {}):
        return "Perturb", participants[pid]["Perturb"][activity]
    if activity in participants[pid].get("ADL", {}):
        return "ADL", participants[pid]["ADL"][activity]
    return None, None

def to_numeric_time(df):
    """Ensure df['time_s'] exists and is numeric; return df or None."""
    if df is None or "time_s" not in df.columns:
        return None
    df = df.copy()
    df["time_s"] = pd.to_numeric(df["time_s"], errors="coerce")
    df = df.dropna(subset=["time_s"])
    return df if len(df) > 1 else None

def plot_acc(ax, df):
    df = to_numeric_time(df)
    if df is None:
        ax.text(0.5, 0.5, "Missing/Bad time_s", ha="center", va="center", transform=ax.transAxes)
        return None
    ax.plot(df["time_s"], df["acc_x"], alpha=0.7, label="acc_x", color="red")
    ax.plot(df["time_s"], df["acc_y"], alpha=0.7, label="acc_y", color="yellow")
    ax.plot(df["time_s"], df["acc_z"], alpha=0.7, label="acc_z", color="blue")
    return (df["time_s"].min(), df["time_s"].max())

def plot_gyro(ax, df):
    df = to_numeric_time(df)
    if df is None:
        ax.text(0.5, 0.5, "Missing/Bad time_s", ha="center", va="center", transform=ax.transAxes)
        return None
    ax.plot(df["time_s"], df["gyro_x"], alpha=0.7, label="gyro_x", color="red")
    ax.plot(df["time_s"], df["gyro_y"], alpha=0.7, label="gyro_y", color="yellow")
    ax.plot(df["time_s"], df["gyro_z"], alpha=0.7, label="gyro_z", color="blue")
    return (df["time_s"].min(), df["time_s"].max())

def plot_ecg(ax, df):
    df = to_numeric_time(df)
    if df is None:
        ax.text(0.5, 0.5, "Missing/Bad time_s", ha="center", va="center", transform=ax.transAxes)
        return None
    ax.plot(df["time_s"], df["ecg_V"], alpha=0.8, color="brown",)
    return (df["time_s"].min(), df["time_s"].max())

def plot_gss(ax, df):
    df = to_numeric_time(df)
    if df is None:
        ax.text(0.5, 0.5, "Missing/Bad time_s", ha="center", va="center", transform=ax.transAxes)
        return None
    ax.plot(df["time_s"], df["gss_V"], alpha=0.8, color="grey",)
    return (df["time_s"].min(), df["time_s"].max())

def highlight_event(ax, labels_df, target_event=4):
    """
    Shade regions where labels_df['event'] == target_event using labels_df['time_s'].
    NOTE: labels_df['time_s'] is in MILLISECONDS, so we convert to SECONDS.
    """
    if labels_df is None:
        return
    if "time_s" not in labels_df.columns or "event" not in labels_df.columns:
        return

    labels_df = labels_df.copy()
    labels_df["time_s"] = pd.to_numeric(labels_df["time_s"], errors="coerce")
    labels_df["event"] = pd.to_numeric(labels_df["event"], errors="coerce")
    labels_df = labels_df.dropna(subset=["time_s", "event"])
    if len(labels_df) == 0:
        return

    # ✅ convert ms -> s
    times_sec = (labels_df["time_s"] / 1000.0).to_numpy()
    events = labels_df["event"].to_numpy()

    mask = (events == target_event)
    if not np.any(mask):
        return

    idx = np.where(mask)[0]
    splits = np.where(np.diff(idx) > 1)[0] + 1
    groups = np.split(idx, splits)

    for g in groups:
        start_t = times_sec[g[0]]
        end_t = times_sec[g[-1]]
        ax.axvspan(start_t, end_t, color="black", alpha=0.4, zorder=0)


# =========================
# 8x3 COMPARISON FIGURE
# =========================
def plot_comparison_grid_8x3(pid, activities, fig_title, out_filename, save=True, show=False, highlight_event_id=4):
    ncols, nrows = 3, 8
    fig, axes = plt.subplots(nrows, ncols, figsize=(22, 18), sharex=False)
    fig.suptitle(f"{pid_str(pid)} - {fig_title}", fontsize=16, fontweight="bold")

    row_titles = [
        "Back Accel", "Back Gyro",
        "Left Thigh Accel", "Left Thigh Gyro",
        "Right Thigh Accel", "Right Thigh Gyro",
        "ECG", "GSS"
    ]

    any_plotted = False

    for col, activity in enumerate(activities):
        kind, data = find_activity(pid, activity)

        # Column title
        axes[0, col].set_title(f"{activity} ({kind if kind else 'Missing'})", fontweight="bold")

        # Row labels (leftmost col)
        for r in range(nrows):
            if col == 0:
                axes[r, col].set_ylabel(row_titles[r], fontweight="bold")

        # If missing activity
        if data is None:
            for r in range(nrows):
                ax = axes[r, col]
                ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
                ax.grid(True, alpha=0.2)
                if r == 7:
                    ax.set_xlabel("Time (s)", fontweight="bold")
            continue

        labels_df = data.get("labels") if kind == "Perturb" else None
        imu = data.get("IMU", {})

        def axr(r): return axes[r, col]

        # Track time range for this column so x-axis stays sane
        tmins, tmaxs = [], []

        # Back Acc (row 0)
        if imu.get("Back") is not None:
            tr = plot_acc(axr(0), imu["Back"])
            if tr: tmins.append(tr[0]); tmaxs.append(tr[1])
            if kind == "Perturb": highlight_event(axr(0), labels_df, target_event=highlight_event_id)
            any_plotted = True
        else:
            axr(0).text(0.5, 0.5, "Missing", ha="center", va="center", transform=axr(0).transAxes)

        # Back Gyro (row 1)
        if imu.get("Back") is not None and "gyro_x" in imu["Back"].columns:
            tr = plot_gyro(axr(1), imu["Back"])
            if tr: tmins.append(tr[0]); tmaxs.append(tr[1])
            if kind == "Perturb": highlight_event(axr(1), labels_df, target_event=highlight_event_id)
            any_plotted = True
        else:
            axr(1).text(0.5, 0.5, "No gyro / Missing", ha="center", va="center", transform=axr(1).transAxes)

        # Left Acc (row 2)
        if imu.get("Left_Thigh") is not None:
            tr = plot_acc(axr(2), imu["Left_Thigh"])
            if tr: tmins.append(tr[0]); tmaxs.append(tr[1])
            if kind == "Perturb": highlight_event(axr(2), labels_df, target_event=highlight_event_id)
            any_plotted = True
        else:
            axr(2).text(0.5, 0.5, "Missing", ha="center", va="center", transform=axr(2).transAxes)

        # Left Gyro (row 3)
        if imu.get("Left_Thigh") is not None and "gyro_x" in imu["Left_Thigh"].columns:
            tr = plot_gyro(axr(3), imu["Left_Thigh"])
            if tr: tmins.append(tr[0]); tmaxs.append(tr[1])
            if kind == "Perturb": highlight_event(axr(3), labels_df, target_event=highlight_event_id)
            any_plotted = True
        else:
            axr(3).text(0.5, 0.5, "No gyro / Missing", ha="center", va="center", transform=axr(3).transAxes)

        # Right Acc (row 4)
        if imu.get("Right_Thigh") is not None:
            tr = plot_acc(axr(4), imu["Right_Thigh"])
            if tr: tmins.append(tr[0]); tmaxs.append(tr[1])
            if kind == "Perturb": highlight_event(axr(4), labels_df, target_event=highlight_event_id)
            any_plotted = True
        else:
            axr(4).text(0.5, 0.5, "Missing", ha="center", va="center", transform=axr(4).transAxes)

        # Right Gyro (row 5)
        if imu.get("Right_Thigh") is not None and "gyro_x" in imu["Right_Thigh"].columns:
            tr = plot_gyro(axr(5), imu["Right_Thigh"])
            if tr: tmins.append(tr[0]); tmaxs.append(tr[1])
            if kind == "Perturb": highlight_event(axr(5), labels_df, target_event=highlight_event_id)
            any_plotted = True
        else:
            axr(5).text(0.5, 0.5, "No gyro / Missing", ha="center", va="center", transform=axr(5).transAxes)

        # ECG (row 6)
        if data.get("ECG") is not None:
            tr = plot_ecg(axr(6), data["ECG"])
            if tr: tmins.append(tr[0]); tmaxs.append(tr[1])
            if kind == "Perturb": highlight_event(axr(6), labels_df, target_event=highlight_event_id)
            any_plotted = True
        else:
            axr(6).text(0.5, 0.5, "Missing", ha="center", va="center", transform=axr(6).transAxes)

        # GSS (row 7)
        if data.get("GSS") is not None:
            tr = plot_gss(axr(7), data["GSS"])
            if tr: tmins.append(tr[0]); tmaxs.append(tr[1])
            if kind == "Perturb": highlight_event(axr(7), labels_df, target_event=highlight_event_id)
            any_plotted = True
        else:
            axr(7).text(0.5, 0.5, "Missing", ha="center", va="center", transform=axr(7).transAxes)

        # Apply consistent x-limits per column (fixes the "flat line / huge axis" issue)
        if len(tmins) > 0 and len(tmaxs) > 0:
            xmin, xmax = float(np.nanmin(tmins)), float(np.nanmax(tmaxs))
            if np.isfinite(xmin) and np.isfinite(xmax) and xmax > xmin:
                for r in range(nrows):
                    axr(r).set_xlim(xmin, xmax)

        # styling
        for r in range(nrows):
            ax = axr(r)
            ax.grid(True, alpha=0.25)
            if r == 7:
                ax.set_xlabel("Time (s)", fontweight="bold")

            if r in [0, 1, 2, 3, 4, 5]:
                handles, labels = ax.get_legend_handles_labels()
                if labels:
                    ax.legend(fontsize=7, loc="upper right")

    plt.tight_layout()

    # highlight legend
    highlight_patch = Patch(color="black", alpha=0.4, label=f"Perturb event == {highlight_event_id}")
    fig.legend(handles=[highlight_patch], loc="upper right", fontsize=10)

    if save and any_plotted:
        out_path = os.path.join(COMPARE_DIR, out_filename)
        fig.savefig(out_path, dpi=300, bbox_inches="tight")
        print(f"💾 Saved: {out_path}")

    if show:
        plt.show()

    plt.close(fig)
    return any_plotted

# =========================
# RUN: 2 FIGURES PER PARTICIPANT
# =========================
print("🎯 Making 2 comparison figures (8x3) per participant")
print("=" * 80)

for pid in sorted(participants.keys()):
    # 1) Sit vs ITR vs ITD
    fig1_name = f"{pid_str(pid)}__8x3__Sit__ITR__ITD.png"
    made1 = plot_comparison_grid_8x3(
        pid,
        ["Sit", "ITR", "ITD"],
        fig_title="Sit (ADL) vs ITR + ITD (Perturb)",
        out_filename=fig1_name,
        save=True,
        show=False,
        highlight_event_id=4
    )

    # 2) Walk vs Jog vs Trip
    fig2_name = f"{pid_str(pid)}__8x3__Walk__Jog__Trip.png"
    made2 = plot_comparison_grid_8x3(
        pid,
        ["Walk", "Jog", "Trip"],
        fig_title="Walk + Jog (ADL) vs Trip (Perturb)",
        out_filename=fig2_name,
        save=True,
        show=False,
        highlight_event_id=4
    )

    if not made1 and not made2:
        print(f"⚠️ {pid_str(pid)}: no matching data for either comparison")

print("=" * 80)
print(f"📁 Done! Saved here:\n{COMPARE_DIR}")

Saving plots to: /Users/chloechristensen/Library/Mobile Documents/com~apple~CloudDocs/School/Y5/BMEG 400K/Exploring_Data/Plots/Comparisons_3col
🎯 Making 2 comparison figures (8x3) per participant
💾 Saved: /Users/chloechristensen/Library/Mobile Documents/com~apple~CloudDocs/School/Y5/BMEG 400K/Exploring_Data/Plots/Comparisons_3col/P1960__8x3__Sit__ITR__ITD.png
💾 Saved: /Users/chloechristensen/Library/Mobile Documents/com~apple~CloudDocs/School/Y5/BMEG 400K/Exploring_Data/Plots/Comparisons_3col/P1960__8x3__Walk__Jog__Trip.png
💾 Saved: /Users/chloechristensen/Library/Mobile Documents/com~apple~CloudDocs/School/Y5/BMEG 400K/Exploring_Data/Plots/Comparisons_3col/P2070__8x3__Sit__ITR__ITD.png
💾 Saved: /Users/chloechristensen/Library/Mobile Documents/com~apple~CloudDocs/School/Y5/BMEG 400K/Exploring_Data/Plots/Comparisons_3col/P2070__8x3__Walk__Jog__Trip.png
💾 Saved: /Users/chloechristensen/Library/Mobile Documents/com~apple~CloudDocs/School/Y5/BMEG 400K/Exploring_Data/Plots/Comparisons_3col/